# Stable Diffusion Turbo - Text-to-Image Generation

![Stable Diffusion](https://stability.ai/favicon.ico)

This example shows how you can use the power of a GPU to quickly generate AI images from text prompts in Saturn Cloud. This code runs on a single GPU of a Jupyter server resource.

This is an example of a text-to-image generation model using Stable Diffusion Turbo, which can create photorealistic or artistic images from natural language descriptions. The model uses a diffusion process that starts with random noise and gradually refines it into a coherent image based on your text prompt. The "Turbo" version is optimized for speed, generating high-quality images in just 1-4 steps instead of the typical 50+ steps.

This notebook creates an interactive Gradio interface where users can type any prompt and instantly see generated images. The interface can be viewed in the notebook or deployed to Saturn Cloud for continuous hosting.

First, we ensure all required packages are installed. This will only install packages that are missing from your environment. We use specific versions of `diffusers`, `transformers`, and `accelerate` to maintain compatibility with the Stable Diffusion Turbo model.

In [ ]:
!pip install -q "diffusers[torch]==0.35.1" transformers==4.56.2 accelerate==1.10.1 gradio safetensors gradio>=4.0.0 pillow>=9.5.0 

This code mainly relies on the Diffusers library for loading the Stable Diffusion model, Gradio for creating the interactive interface, and PyTorch for GPU acceleration. The architecture and components are automatically handled.

In [ ]:
import torch
from diffusers import AutoPipelineForText2Image
import gradio as gr
from PIL import Image
import numpy as np
import matplotlib.pyplot as plt

We load the SDXL-Turbo model from Hugging Face Hub, which is optimized for fast single-step image generation. The model is approximately 7GB and will be downloaded on first run.

In [ ]:
print("Loading Stable Diffusion Turbo model...")
print("(This may take a few minutes on first run as the model downloads)\n")

# Load the model with optimizations for GPU
pipe = AutoPipelineForText2Image.from_pretrained(
    "stabilityai/sdxl-turbo",
    torch_dtype=torch.float16,
    variant="fp16"
)

# Move model to GPU
pipe = pipe.to("cuda")

pipe.enable_attention_slicing()

print("✓ Model loaded successfully!")

Here we define the core function that interacts with the loaded pipeline. The `generate_image` function takes a text prompt and uses the diffusion model to create an image. For the Turbo model, we use just 1 inference step and a guidance scale of 0.0 for maximum speed, as this is its intended configuration. The function also supports a random seed for reproducible results.

In [ ]:
def generate_image(prompt, num_inference_steps=1, guidance_scale=0.0, seed=None):
    """
    Generate an image from a text prompt.

    Args:
        prompt: Text description of the desired image
        num_inference_steps: Number of denoising steps (1 for Turbo)
        guidance_scale: How strictly to follow the prompt (0.0 for Turbo)
        seed: Random seed for reproducibility (None for random)

    Returns:
        PIL Image
    """
    # Set seed for reproducibility if provided
    generator = None
    device = pipe.device # Define the device
    if seed is not None:
        generator = torch.Generator(device=device).manual_seed(seed)

    # Generate image
    image = pipe(
        prompt=prompt,
        num_inference_steps=num_inference_steps,
        guidance_scale=guidance_scale,
        generator=generator
    ).images[0]

    return image

To showcase the model's ability to create variations, we define a function to generate a grid of images from the same prompt using different random seeds. This is useful for exploring different interpretations of a single concept. The function creates a matplotlib figure displaying all generated images in a grid layout with their corresponding seeds.

In [ ]:
# Gradio Blocks Interface for Image Grid Generation
def generate_image_grid_gradio(prompt, num_images=4, seed_start=42):
    """
    Generate a grid of images for Gradio interface.
    
    Args:
        prompt: Text description
        num_images: Number of variations to generate
        seed_start: Starting seed
    
    Returns:
        matplotlib figure
    """
    images = []
    seeds = []
    
    for i in range(num_images):
        seed = seed_start + i
        image = generate_image(prompt, seed=seed)
        images.append(image)
        seeds.append(seed)
    
    # Create grid layout
    grid_size = int(np.ceil(np.sqrt(num_images)))
    fig, axes = plt.subplots(grid_size, grid_size, figsize=(10, 10))
    
    if grid_size == 1:
        axes = [[axes]]
    elif len(axes.shape) == 1:
        axes = [axes]
    
    fig.suptitle(f'Prompt: "{prompt}"', fontsize=14, fontweight='bold')
    
    for idx, ax in enumerate(axes.flat):
        if idx < len(images):
            ax.imshow(images[idx])
            ax.set_title(f'Seed: {seeds[idx]}', fontsize=8)
        ax.axis('off')
    
    plt.tight_layout()
    return fig

Now let's create an advanced interactive web interface using Gradio's Blocks API. This interface provides more control, allowing users to generate a grid of image variations from a single prompt. It includes sliders to control the number of images and the starting seed for reproducibility.

In [ ]:
# Create Gradio Blocks interface
with gr.Blocks(title="🎨 AI Image Grid Generator") as demo:
    gr.Markdown("# 🎨 AI Image Grid Generator")
    gr.Markdown("Generate multiple variations of the same prompt with different seeds")
    
    with gr.Row():
        with gr.Column():
            prompt_input = gr.Textbox(
                label="Prompt",
                placeholder="Describe the image you want to generate...",
                lines=2
            )
            num_images = gr.Slider(
                minimum=1,
                maximum=9,
                value=4,
                step=1,
                label="Number of Images"
            )
            seed_start = gr.Slider(
                minimum=0,
                maximum=1000,
                value=42,
                step=1,
                label="Starting Seed"
            )
            generate_btn = gr.Button("Generate Image Grid", variant="primary")
        
        with gr.Column():
            output_plot = gr.Plot(label="Generated Images")
    
    examples = gr.Examples(
        examples=[
            ["a cat wearing sunglasses on a beach", 4, 42],
            ["a futuristic city at sunset with flying cars", 4, 123],
            ["a magical forest with glowing mushrooms", 4, 456]
        ],
        inputs=[prompt_input, num_images, seed_start]
    )
    
    generate_btn.click(
        fn=generate_image_grid_gradio,
        inputs=[prompt_input, num_images, seed_start],
        outputs=output_plot
    )

print("✓ Gradio Grid interface created!")
demo.launch(share=False)

In [ ]:
def gradio_generate(prompt, seed, num_steps):
    """
    Wrapper function for Gradio interface.
    
    Args:
        prompt: Text description
        seed: Random seed (-1 for random)
        num_steps: Number of inference steps (1-4 for Turbo)
    
    Returns:
        PIL Image
    """
    # Convert -1 to None for random seed
    actual_seed = None if seed == -1 else seed
    
    # Generate and return image
    return generate_image(prompt, num_inference_steps=num_steps, seed=actual_seed)


# Create Gradio interface
demo = gr.Interface(
    fn=gradio_generate,
    inputs=[
        gr.Textbox(
            label="Prompt",
            placeholder="Describe the image you want to generate...",
            lines=3
        ),
        gr.Slider(
            minimum=-1,
            maximum=10000,
            value=-1,
            step=1,
            label="Seed (-1 for random)"
        ),
        gr.Slider(
            minimum=1,
            maximum=4,
            value=1,
            step=1,
            label="Inference Steps (1 = fastest, 4 = highest quality)"
        )
    ],
    outputs=gr.Image(label="Generated Image", type="pil"),
    title="🎨 Stable Diffusion Turbo - Text-to-Image Generator",
    description="Generate AI images from text prompts using Stable Diffusion Turbo. Type your prompt and click Submit!",
    examples=[
        ["a cat wearing sunglasses on a beach", 42, 1],
        ["a futuristic city at sunset with flying cars", 123, 1],
        ["a magical forest with glowing mushrooms and fairy lights", 456, 1],
        ["portrait of a wise old wizard with a long beard", 789, 1],
        ["a steampunk robot playing piano in a Victorian mansion", 999, 1],
        ["an underwater city with bioluminescent coral and fish", 555, 1]
    ],
    theme=gr.themes.Soft(),
    allow_flagging="never"
)

print("\n✓ Gradio interface created!")
print("  Run the next cell to launch the interface.")

### Launch the Interface

Run this cell to launch the interactive Gradio interface. You can use it directly in the notebook or share the public URL. The interface connects the UI elements to the model, allowing real-time image generation.

In [ ]:
# Launch the interface
demo.launch(share=False, debug=False)

The interface is now live! You can:
- Type any prompt in the text box
- Adjust the seed for reproducibility
- Change inference steps (1 = fastest, 4 = best quality)
- Click example prompts to try them instantly
- Generate unlimited images!

**Tip:** For best results, be descriptive in your prompts. Instead of "a cat", try "a fluffy orange cat sitting on a windowsill at sunset".